# Volatility-Based Position Sizing
Scale position size inversely with realized volatility. 
The idea: take bigger positions in calm markets, smaller in volatile.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('dark_background')

df = yf.download('SPY', start='2018-01-01', end='2024-12-31', progress=False)
df['returns'] = df['Close'].pct_change()
df['vol_20'] = df['returns'].rolling(20).std() * np.sqrt(252)
df['vol_60'] = df['returns'].rolling(60).std() * np.sqrt(252)
df = df.dropna()

In [ ]:
# Volatility targeting: target 15% annual vol
TARGET_VOL = 0.15

df['vol_scalar'] = (TARGET_VOL / df['vol_20']).clip(0.2, 2.0)
df['scaled_returns'] = df['returns'] * df['vol_scalar'].shift(1)  # use yesterday's scalar

# Equity curves
df['raw_equity'] = 100_000 * (1 + df['returns']).cumprod()
df['scaled_equity'] = 100_000 * (1 + df['scaled_returns']).cumprod()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[2, 1])

ax1.plot(df.index, df['raw_equity'], label='Unscaled (Buy & Hold)', color='#71717a', linewidth=0.8)
ax1.plot(df.index, df['scaled_equity'], label='Vol-Targeted (15%)', color='#22d3ee', linewidth=1.2)
ax1.set_title('Volatility-Targeted Position Sizing', fontsize=14)
ax1.legend()
ax1.grid(alpha=0.2)

ax2.fill_between(df.index, df['vol_20'] * 100, alpha=0.3, color='#f97316')
ax2.axhline(TARGET_VOL * 100, color='#22d3ee', linestyle='--', label=f'Target ({TARGET_VOL*100:.0f}%)')
ax2.set_title('20-Day Realized Volatility (annualized)', fontsize=12)
ax2.set_ylabel('Volatility %')
ax2.legend()
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Compare risk metrics
raw_ret = df['returns']
scaled_ret = df['scaled_returns']

for name, r in [('Unscaled', raw_ret), ('Vol-Targeted', scaled_ret)]:
    sharpe = r.mean() / r.std() * np.sqrt(252)
    cum = (1 + r).cumprod()
    max_dd = ((cum - cum.cummax()) / cum.cummax()).min()
    realized_vol = r.std() * np.sqrt(252)
    print(f'{name:15s} | Sharpe: {sharpe:.2f} | MaxDD: {max_dd:.1%} | Vol: {realized_vol:.1%}')